# Pernod Ricard Japan — Supplier Intelligence Hub

## AI-Powered Supplier Data Standardization on Databricks

---

**Business Context:** Pernod Ricard Japan manages **500+ suppliers** across Japan and Asia — raw materials, glass bottles, packaging, logistics, brewing equipment, and more. Supplier data arrives from multiple sources in inconsistent formats:

- **Mixed languages**: Japanese names (`山田ガラス工業株式会社`) and English names (`Yamada Glass Industries Co., Ltd.`)
- **Inconsistent categories**: `ガラスびん`, `Glass Bottles`, `glass bottle`, `ボトル` — all mean the same thing
- **Messy addresses**: Mix of Japanese postal format (`〒136-0071 東京都江東区...`) and English format
- **Different currency notations**: `JPY`, `円`, `yen` used interchangeably
- **Missing certifications**: Some suppliers lack required food safety certifications (ISO 22000, HACCP)

**Manual standardization is time-consuming, error-prone, and doesn't scale.**

This demo shows how Databricks solves this end-to-end with AI — from raw data ingestion to a fully operational procurement intelligence platform.

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│                    PERNOD RICARD JAPAN                              │
│               Supplier Intelligence Hub                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Layer 1: AI SQL Cleaning                                          │
│  ┌──────────┐    ┌──────────────┐    ┌─────────────────┐           │
│  │ Raw Data │───>│ ai_query()   │───>│ Cleaned Tables  │           │
│  │ 500 rows │    │ Categories   │    │ Standardized    │           │
│  │ JP/EN    │    │ Addresses    │    │ Deduplicated    │           │
│  └──────────┘    │ Duplicates   │    └────────┬────────┘           │
│                  └──────────────┘             │                     │
│  Layer 2: Knowledge Assistant                 │                     │
│  ┌──────────────┐    ┌───────────────┐       │                     │
│  │ Policy Docs  │───>│ Vector Search │       │                     │
│  │ Compliance   │    │ RAG Index     │       │                     │
│  │ Scorecards   │    └───────┬───────┘       │                     │
│  └──────────────┘            │               │                     │
│  Layer 3: Agent Processing   │               │                     │
│  ┌───────────────────────────┴───────────────┴──┐                  │
│  │ Rule-Based Anomaly Detection                  │                  │
│  │ ┌──────────┐ ┌──────────┐ ┌───────────────┐  │                  │
│  │ │ Price    │ │ Lead Time│ │ Certification │  │                  │
│  │ │ Anomaly  │ │ Anomaly  │ │ Missing       │  │                  │
│  │ └────┬─────┘ └────┬─────┘ └──────┬────────┘  │                  │
│  │      └────────────┼──────────────┘            │                  │
│  │           ┌───────┴───────┐                   │                  │
│  │           │ Auto-Approve  │                   │                  │
│  │           │ or Escalate   │                   │                  │
│  │           └───┬───────┬───┘                   │                  │
│  └───────────────┼───────┼───────────────────────┘                  │
│          ┌───────┘       └────────┐                                 │
│          ▼                        ▼                                 │
│  ┌──────────────┐     ┌──────────────────┐                         │
│  │ Gold Table   │     │ Escalation Queue │                         │
│  │ 118 approved │     │ 382 to review    │                         │
│  └──────────────┘     └──────────────────┘                         │
│                                                                     │
│  Layer 4: Business Surfaces                                        │
│  ┌────────┐ ┌────┐ ┌────────────┐ ┌──────────────┐ ┌─────┐       │
│  │  App   │ │ KA │ │ Supervisor │ │ Custom Agent │ │Genie│       │
│  └────────┘ └────┘ └────────────┘ └──────────────┘ └─────┘       │
└─────────────────────────────────────────────────────────────────────┘
```

---
# Layer 1: AI SQL Cleaning with `ai_query()`

Databricks AI SQL Functions let us call foundation models **directly from SQL**. No Python, no notebooks, no external APIs — just SQL.

We use `ai_query()` to:
1. **Standardize 500 suppliers into 10 product categories** — the AI understands both Japanese (`ガラスびん`) and English (`Glass Bottles`)
2. **Parse free-text addresses** into structured fields (postal code, prefecture, city)
3. **Detect 444 duplicate pairs** across languages — matching `札幌フィルター工業` to `Sapporo Filter Industries`

### 1.1 — The Raw Data Problem
Let's look at what arrives from supplier onboarding systems:

In [ ]:
-- Look at raw supplier data — notice the inconsistencies
SELECT supplier_name, category, address, certification, currency
FROM opm_catalog.supplier_hub.raw_suppliers
LIMIT 20

In [ ]:
-- How many different category names exist in the raw data?
SELECT category, COUNT(*) as cnt
FROM opm_catalog.supplier_hub.raw_suppliers
GROUP BY category
ORDER BY cnt DESC

### 1.2 — AI-Standardized Categories
After running `ai_query()`, all category variations collapse into **10 clean categories**:

In [ ]:
-- After AI standardization: 10 clean categories from dozens of variations
SELECT standardized_category, COUNT(*) as supplier_count
FROM opm_catalog.supplier_hub.suppliers_cleaned
GROUP BY standardized_category
ORDER BY supplier_count DESC

### 1.3 — AI Duplicate Detection Across Languages
The AI matched supplier names across Japanese and English — something no rule-based system could do:

In [ ]:
-- AI detected duplicates across Japanese ↔ English names
SELECT name_a, name_b, is_duplicate
FROM opm_catalog.supplier_hub.duplicate_candidates
WHERE TRIM(is_duplicate) = 'YES'
ORDER BY name_a
LIMIT 15

In [ ]:
-- Total duplicate pairs found
SELECT COUNT(*) as total_duplicate_pairs
FROM opm_catalog.supplier_hub.duplicate_candidates
WHERE TRIM(is_duplicate) = 'YES'

### 1.4 — Cleaned & Structured Output
The final cleaned table has structured addresses, standardized categories, and normalized currencies:

In [ ]:
-- Clean, structured supplier data
SELECT supplier_id, supplier_name, standardized_category, 
       prefecture_jp, city, postal_code,
       certification, unit_price, currency_standardized, 
       reliability_score, lead_time_days
FROM opm_catalog.supplier_hub.suppliers_cleaned
LIMIT 20

---
# Layer 2: Knowledge Assistant (RAG over Procurement Docs)

Three procurement policy documents were indexed using **Databricks Vector Search**:

| Document | Content |
|----------|---------|
| `supplier_policy.md` | Supplier qualification rules, certification requirements, pricing limits |
| `compliance_checklist.md` | Japan-specific compliance: 食品衛生法 (Food Safety Act), 酒税法 (Liquor Tax Act) |
| `scorecard_methodology.md` | Supplier scoring criteria, audit frequency, reliability thresholds |

A **Knowledge Assistant** (Agent Bricks) provides RAG-grounded answers with source citations.

**Endpoint:** `ka-94303de4-endpoint`

Example questions:
- "What certifications are mandatory for food-contact suppliers?"
- "What are the acceptable price ranges for glass bottles?"
- "What is the audit frequency for Tier 1 suppliers?"
- "What are the Japan-specific compliance requirements?"

---
# Layer 3: Agent Processing — Auto-Approve or Escalate

Rule-based anomaly detection flags suppliers for 5 types of issues:

| Check | Rule | Flag |
|-------|------|------|
| **Price Anomaly** | Unit price > 2x category median | `PRICE_ANOMALY` |
| **Lead Time** | Lead time > 45 days | `LEAD_TIME_ANOMALY` |
| **Missing Certification** | Food-contact categories without ISO 22000/FSSC/HACCP | `MISSING_CERTIFICATION` |
| **Low Reliability** | Reliability score < 70 | `LOW_RELIABILITY` |
| **Missing Audit** | No audit in the last 12 months | `MISSING_AUDIT` |

Suppliers with **zero flags** are auto-approved to the **Gold Table**. The rest go to the **Escalation Queue** with risk levels (HIGH/MEDIUM/LOW).

In [ ]:
-- Auto-approved suppliers (Gold Table) — clean, compliant, no issues
SELECT COUNT(*) as auto_approved_count FROM opm_catalog.supplier_hub.gold_suppliers

In [ ]:
-- Escalation queue breakdown by risk level
SELECT risk_level, COUNT(*) as supplier_count
FROM opm_catalog.supplier_hub.escalation_queue
GROUP BY risk_level
ORDER BY CASE risk_level WHEN 'HIGH' THEN 1 WHEN 'MEDIUM' THEN 2 ELSE 3 END

In [ ]:
-- Sample escalated suppliers with their issues
SELECT supplier_id, supplier_name, category, risk_level, issues, recommendation
FROM opm_catalog.supplier_hub.escalation_queue
WHERE risk_level = 'HIGH'
ORDER BY supplier_name
LIMIT 10

In [ ]:
-- Gold table: approved suppliers ready for procurement
SELECT supplier_id, supplier_name, standardized_category, 
       prefecture_jp, certification, reliability_score, 
       unit_price, currency_standardized, lead_time_days
FROM opm_catalog.supplier_hub.gold_suppliers
ORDER BY reliability_score DESC
LIMIT 15

---
# Layer 4: Unity Catalog Functions as AI Agent Tools

Five SQL functions are registered in Unity Catalog and can be called by AI agents directly from the Playground:

| Function | Purpose |
|----------|---------|
| `get_supplier_overview(supplier_id)` | Full supplier profile with delivery metrics |
| `list_non_compliant_suppliers()` | All escalated suppliers by risk level |
| `get_spend_by_category()` | Spend breakdown across product categories |
| `validate_delivery_quality(supplier_id)` | Audit a supplier's delivery performance |
| `get_supplier_risk_summary()` | Portfolio-level risk exposure |

In [ ]:
-- UC Function: Get a full supplier overview
SELECT * FROM opm_catalog.supplier_hub.get_supplier_overview('SUP-0001')

In [ ]:
-- UC Function: List non-compliant suppliers
SELECT * FROM opm_catalog.supplier_hub.list_non_compliant_suppliers() LIMIT 10

In [ ]:
-- UC Function: Spend breakdown by category
SELECT * FROM opm_catalog.supplier_hub.get_spend_by_category()

In [ ]:
-- UC Function: Validate delivery quality for a specific supplier
SELECT * FROM opm_catalog.supplier_hub.validate_delivery_quality('SUP-0001')

In [ ]:
-- UC Function: Portfolio risk summary
SELECT * FROM opm_catalog.supplier_hub.get_supplier_risk_summary()

---
# Layer 4: Business Surfaces

Four AI-powered interfaces built on Databricks:

### 1. Knowledge Assistant (Agent Bricks)
- **Endpoint:** `ka-94303de4-endpoint`
- No-code RAG over 3 procurement policy documents
- Vector Search managed embedding index
- Grounded answers with source citations

### 2. Custom Agent (MLflow 3 + LangGraph)
- **Endpoint:** `supplier-intelligence-agent`
- Code-based agent combining Vector Search RAG + 5 UC Function tools
- Built with `ResponsesAgent` pattern in MLflow 3
- Can answer policy questions AND query live data in the same turn

### 3. Supervisor Agent (Agent Bricks)
- **Endpoint:** `mas-f1bf4d08-endpoint`
- Multi-agent orchestration routing to KA + Genie Space
- Automatically decides which agent to call based on the question

### 4. Genie Space
- **Space ID:** `01f118869d191f1f97ee38c28c8b2c4c`
- Natural language SQL over supplier tables
- Ask "Which suppliers in Tokyo have the highest reliability?" and get instant results

### 5. Databricks App
- **URL:** `https://pernod-supplier-hub-7474649123061285.aws.databricksapps.com`
- React/Node.js full-stack application
- Dashboard with KPIs, Knowledge Assistant chat, Review Queue, Pipeline Trigger

---
# Summary: Key Databricks Features Demonstrated

| Feature | How We Used It |
|---------|---------------|
| **AI SQL Functions (`ai_query()`)** | Standardize categories, parse addresses, detect duplicates — all in SQL |
| **Agent Bricks (KA)** | No-code Knowledge Assistant with RAG over procurement docs |
| **Agent Bricks (Supervisor)** | Multi-agent orchestration routing to KA + Genie |
| **Unity Catalog Functions** | 5 SQL functions registered as AI agent tools, callable from Playground |
| **Vector Search** | Managed embedding index over policy documents for semantic retrieval |
| **MLflow 3 + LangGraph** | Custom code-based agent with `ResponsesAgent` pattern |
| **Genie Space** | Natural language SQL exploration over supplier tables |
| **Databricks Apps** | Full-stack web application with SP OAuth authentication |
| **Serverless Compute** | Everything runs serverless — no cluster management |
| **Unity Catalog** | Governance, lineage, and permissions across all data and AI assets |

**Everything runs on a single platform — no stitching tools together.**